# Setup, load real OpenFDA chunks with entities

In [1]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(message)s")

from medrag.processing.storage import load_chunks
from medrag.ner.ner import extract_medical_entities

CHUNKS_DIR = str(PROJECT_ROOT / "data" / "processed" / "chunks")

# Grab a handful of real Metformin chunks - likely to include an
# "INDICATIONS AND USAGE" style section explicitly stating what it treats
metformin_chunks = [
    c for c in load_chunks(source="openfda", topic="diabetes", output_dir=CHUNKS_DIR)
    if "metformin" in c.chunk_id.lower()
][:8]

print(f"Found {len(metformin_chunks)} Metformin chunks")
for chunk in metformin_chunks:
    result = extract_medical_entities(chunk.raw_text)
    print(f"\nchunk_id={chunk.chunk_id}")
    print(f"  chemicals: {result['chemicals']}")
    print(f"  diseases: {result['diseases']}")
    print(f"  text preview: {chunk.raw_text[:200]}")

Project root: C:\Users\DELL\Desktop\medrag


Loading NER model 'en_ner_bc5cdr_md'...


Found 8 Metformin chunks


c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]



chunk_id=Metformin Hydrochloride_openfda_0
  chemicals: ['Metformin hydrochloride', 'Metformin hydrochloride']
  diseases: ['type 2 diabetes mellitus', 'type 2 diabetes mellitus']
  text preview: 1 INDICATIONS & USAGE Metformin hydrochloride tablets are indicated as an adjunct to diet and exercise to improve glycemic control in adults and pediatric patients 10 years of age and older with type 

chunk_id=Metformin Hydrochloride_openfda_1
  chemicals: ['metformin hydrochloride', 'metformin hydrochloride', 'Discontinue', 'Metformin Hydrochloride', 'metformin hydrochloride']
  diseases: []
  text preview: 2 DOSAGE & ADMINISTRATION Adult Dosage for metformin hydrochloride tablets: Starting dose: 500 mg orally twice a day or 850 mg once a day, with meals ( 2.1 ) Increase the dose in increments of 500 mg 

chunk_id=Metformin Hydrochloride_openfda_2
  chemicals: ['metformin hydrochloride', 'metformin hydrochloride', 'metformin hydrochloride', 'Metformin hydrochloride', 'metformin hydrochlorid

# Check chunk metadata for section info

In [2]:
for chunk in metformin_chunks[:3]:
    print(f"chunk_id={chunk.chunk_id}")
    print(f"  metadata: {chunk.metadata}")
    print()

chunk_id=Metformin Hydrochloride_openfda_0
  metadata: {'field': 'indications_and_usage'}

chunk_id=Metformin Hydrochloride_openfda_1
  metadata: {'field': 'dosage_and_administration'}

chunk_id=Metformin Hydrochloride_openfda_2
  metadata: {'field': 'dosage_and_administration'}



# Enumerate all distinct field values across OpenFDA chunks

In [3]:
from collections import Counter

all_openfda_chunks = list(load_chunks(source="openfda", topic="diabetes", output_dir=CHUNKS_DIR))
field_counts = Counter(c.metadata.get("field", "MISSING") for c in all_openfda_chunks)

for field, count in field_counts.most_common():
    print(f"  {field}: {count}")

  adverse_reactions: 140
  warnings_and_cautions: 95
  drug_interactions: 89
  dosage_and_administration: 68
  indications_and_usage: 61
  mechanism_of_action: 29
  contraindications: 25


# Build the field-to-relationship mapping and extraction function

In [4]:
from medrag.ner.ner import extract_medical_entities

FIELD_TO_RELATIONSHIP = {
    "indications_and_usage": "TREATS",
    "contraindications": "CONTRAINDICATED_IN",
    "adverse_reactions": "CAUSES",
    "warnings_and_cautions": "CAUSES",
}
# mechanism_of_action, dosage_and_administration, drug_interactions:
# deliberately excluded (see reasoning above) - not mapped, chunks with
# these fields simply produce no relationships.

def extract_relationships_from_chunk(chunk) -> list:
    """For OpenFDA chunks only: map the chunk's labeled section (field
    metadata) directly to a relationship type, using entities extracted
    from the chunk text. Chunks whose field isn't in
    FIELD_TO_RELATIONSHIP produce no relationships (deliberate scope
    exclusion, not an oversight)."""
    field = chunk.metadata.get("field")
    relationship_type = FIELD_TO_RELATIONSHIP.get(field)
    if relationship_type is None:
        return []

    entities = extract_medical_entities(chunk.raw_text)
    relationships = []
    for chem in set(entities["chemicals"]):
        for disease in set(entities["diseases"]):
            relationships.append({
                "chemical": chem,
                "relationship": relationship_type,
                "disease": disease,
                "source_chunk_id": chunk.chunk_id,
                "field": field,
            })
    return relationships


# Test on our 8 real Metformin chunks
all_relationships = []
for chunk in metformin_chunks:
    rels = extract_relationships_from_chunk(chunk)
    all_relationships.extend(rels)

print(f"Total relationships extracted: {len(all_relationships)}")
for r in all_relationships:
    print(f"  ({r['chemical']}) --[{r['relationship']}]--> ({r['disease']})   [from {r['field']}]")

Total relationships extracted: 83
  (Metformin hydrochloride) --[TREATS]--> (type 2 diabetes mellitus)   [from indications_and_usage]
  (metformin) --[CONTRAINDICATED_IN]--> (coma)   [from contraindications]
  (metformin) --[CONTRAINDICATED_IN]--> (metabolic acidosis)   [from contraindications]
  (metformin) --[CONTRAINDICATED_IN]--> (Hypersensitivity)   [from contraindications]
  (metformin) --[CONTRAINDICATED_IN]--> (diabetic ketoacidosis)   [from contraindications]
  (metformin) --[CONTRAINDICATED_IN]--> (renal impairment)   [from contraindications]
  (Metformin hydrochloride) --[CONTRAINDICATED_IN]--> (coma)   [from contraindications]
  (Metformin hydrochloride) --[CONTRAINDICATED_IN]--> (metabolic acidosis)   [from contraindications]
  (Metformin hydrochloride) --[CONTRAINDICATED_IN]--> (Hypersensitivity)   [from contraindications]
  (Metformin hydrochloride) --[CONTRAINDICATED_IN]--> (diabetic ketoacidosis)   [from contraindications]
  (Metformin hydrochloride) --[CONTRAINDICATED

# Confirm chunk.source_id gives us the drug name directly

In [5]:
for chunk in metformin_chunks[:3]:
    print(f"chunk_id={chunk.chunk_id}")
    print(f"  source_id={chunk.source_id}")
    print(f"  topics={chunk.topics}")

chunk_id=Metformin Hydrochloride_openfda_0
  source_id=Metformin Hydrochloride
  topics=['diabetes']
chunk_id=Metformin Hydrochloride_openfda_1
  source_id=Metformin Hydrochloride
  topics=['diabetes']
chunk_id=Metformin Hydrochloride_openfda_2
  source_id=Metformin Hydrochloride
  topics=['diabetes']


# Rebuild: known drug (from source_id) × extracted diseases only

In [6]:
def extract_relationships_from_chunk_v2(chunk) -> list:
    """For OpenFDA chunks only: the drug is already known with certainty
    from chunk.source_id (every chunk in a drug's label document shares
    the same source_id, set during Phase 5 chunking) - no need to
    extract or guess the chemical from text. Only the disease side needs
    NER. This avoids the cartesian-product noise of pairing every
    extracted CHEMICAL with every extracted DISEASE (which produced
    nonsense pairings like Vitamin-B-causes-lactic-acidosis from
    unrelated chemicals mentioned in the same passage), and avoids
    fragmenting one drug into multiple node identities based on how the
    text happened to phrase its name (metformin / Metformin / Metformin
    hydrochloride)."""
    field = chunk.metadata.get("field")
    relationship_type = FIELD_TO_RELATIONSHIP.get(field)
    if relationship_type is None:
        return []

    drug_name = chunk.source_id
    entities = extract_medical_entities(chunk.raw_text)

    relationships = []
    for disease in set(entities["diseases"]):
        relationships.append({
            "chemical": drug_name,
            "relationship": relationship_type,
            "disease": disease,
            "source_chunk_id": chunk.chunk_id,
            "field": field,
        })
    return relationships


all_relationships_v2 = []
for chunk in metformin_chunks:
    rels = extract_relationships_from_chunk_v2(chunk)
    all_relationships_v2.extend(rels)

print(f"Total relationships extracted: {len(all_relationships_v2)}")
for r in all_relationships_v2:
    print(f"  ({r['chemical']}) --[{r['relationship']}]--> ({r['disease']})   [from {r['field']}]")

Total relationships extracted: 25
  (Metformin Hydrochloride) --[TREATS]--> (type 2 diabetes mellitus)   [from indications_and_usage]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (coma)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (metabolic acidosis)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (Hypersensitivity)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (diabetic ketoacidosis)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (renal impairment)   [from contraindications]
  (Metformin Hydrochloride) --[CAUSES]--> (myalgias)   [from warnings_and_cautions]
  (Metformin Hydrochloride) --[CAUSES]--> (hypotension)   [from warnings_and_cautions]
  (Metformin Hydrochloride) --[CAUSES]--> (metformin-associated lactic acidosis)   [from warnings_and_cautions]
  (Metformin Hydrochloride) --[CAUSES]--> (abdominal pain)   [from warnings_and_cautions

# Normalize entity names before building relationships

In [7]:
def normalize_entity(name: str) -> str:
    """Normalize an entity name for deduplication/graph-node identity.
    Case differences (Hypoglycemia vs hypoglycemia) are a text-casing
    artifact, not a different real-world entity."""
    return name.strip().lower()


def extract_relationships_from_chunk_v3(chunk) -> list:
    field = chunk.metadata.get("field")
    relationship_type = FIELD_TO_RELATIONSHIP.get(field)
    if relationship_type is None:
        return []

    drug_name = chunk.source_id
    entities = extract_medical_entities(chunk.raw_text)

    # dedupe by normalized form, keep first-seen original casing for display
    seen_normalized = {}
    for disease in entities["diseases"]:
        norm = normalize_entity(disease)
        if norm not in seen_normalized:
            seen_normalized[norm] = disease

    relationships = []
    for norm_disease, original_disease in seen_normalized.items():
        relationships.append({
            "chemical": drug_name,
            "chemical_normalized": normalize_entity(drug_name),
            "relationship": relationship_type,
            "disease": original_disease,
            "disease_normalized": norm_disease,
            "source_chunk_id": chunk.chunk_id,
            "field": field,
        })
    return relationships


all_relationships_v3 = []
for chunk in metformin_chunks:
    rels = extract_relationships_from_chunk_v3(chunk)
    all_relationships_v3.extend(rels)

print(f"Total relationships extracted: {len(all_relationships_v3)}")
for r in all_relationships_v3:
    print(f"  ({r['chemical']}) --[{r['relationship']}]--> ({r['disease']})   [from {r['field']}]")

Total relationships extracted: 23
  (Metformin Hydrochloride) --[TREATS]--> (type 2 diabetes mellitus)   [from indications_and_usage]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (renal impairment)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (Hypersensitivity)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (metabolic acidosis)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (diabetic ketoacidosis)   [from contraindications]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (coma)   [from contraindications]
  (Metformin Hydrochloride) --[CAUSES]--> (Lactic Acidosis)   [from warnings_and_cautions]
  (Metformin Hydrochloride) --[CAUSES]--> (Hypoglycemia)   [from warnings_and_cautions]
  (Metformin Hydrochloride) --[CAUSES]--> (metformin-associated lactic acidosis)   [from warnings_and_cautions]
  (Metformin Hydrochloride) --[CAUSES]--> (myalgias)   [from warnings_and_cautio

# Final cross-chunk aggregation, keyed by (drug, relationship, disease)

In [8]:
from collections import defaultdict

def aggregate_relationships(all_relationships: list) -> list:
    """Consolidate relationships across ALL chunks (not just within one),
    keyed by (chemical_normalized, relationship, disease_normalized).
    Multiple chunks asserting the same fact become one edge with
    multiple source_chunk_ids as evidence - stronger support for a fact
    stated repeatedly, and a free citation trail back to source text."""
    grouped = defaultdict(lambda: {"source_chunk_ids": [], "display_disease": None, "display_chemical": None})

    for r in all_relationships:
        key = (r["chemical_normalized"], r["relationship"], r["disease_normalized"])
        entry = grouped[key]
        entry["source_chunk_ids"].append(r["source_chunk_id"])
        if entry["display_disease"] is None:
            entry["display_disease"] = r["disease"]
            entry["display_chemical"] = r["chemical"]

    result = []
    for (chem_norm, rel, dis_norm), entry in grouped.items():
        result.append({
            "chemical": entry["display_chemical"],
            "chemical_normalized": chem_norm,
            "relationship": rel,
            "disease": entry["display_disease"],
            "disease_normalized": dis_norm,
            "source_chunk_ids": entry["source_chunk_ids"],
        })
    return result


final_relationships = aggregate_relationships(all_relationships_v3)

print(f"Final unique relationships: {len(final_relationships)}")
for r in final_relationships:
    evidence_count = len(r["source_chunk_ids"])
    print(f"  ({r['chemical']}) --[{r['relationship']}]--> ({r['disease']})   [{evidence_count} chunk(s)]")

Final unique relationships: 19
  (Metformin Hydrochloride) --[TREATS]--> (type 2 diabetes mellitus)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (renal impairment)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (Hypersensitivity)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (metabolic acidosis)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (diabetic ketoacidosis)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CONTRAINDICATED_IN]--> (coma)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CAUSES]--> (Lactic Acidosis)   [2 chunk(s)]
  (Metformin Hydrochloride) --[CAUSES]--> (Hypoglycemia)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CAUSES]--> (metformin-associated lactic acidosis)   [3 chunk(s)]
  (Metformin Hydrochloride) --[CAUSES]--> (myalgias)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CAUSES]--> (abdominal pain)   [1 chunk(s)]
  (Metformin Hydrochloride) --[CAUSES]--> (respiratory distress) 

# Connect to Neo4j from Python, confirm connectivity

In [9]:
# run once if not already installed:
# pip install neo4j --break-system-packages

from neo4j import GraphDatabase
from config.settings import settings

driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password),
)

driver.verify_connectivity()
print("Connected to Neo4j successfully")

Connected to Neo4j successfully


# Create constraints, write the test relationships to Neo4j

In [10]:
def setup_constraints(driver):
    with driver.session() as session:
        session.run("CREATE CONSTRAINT drug_name IF NOT EXISTS FOR (d:Drug) REQUIRE d.normalized_name IS UNIQUE")
        session.run("CREATE CONSTRAINT disease_name IF NOT EXISTS FOR (dis:Disease) REQUIRE dis.normalized_name IS UNIQUE")
    print("Constraints created")


def write_relationships(driver, relationships: list):
    with driver.session() as session:
        for r in relationships:
            session.run(
                f"""
                MERGE (d:Drug {{normalized_name: $chem_norm}})
                ON CREATE SET d.name = $chem_display
                MERGE (dis:Disease {{normalized_name: $dis_norm}})
                ON CREATE SET dis.name = $dis_display
                MERGE (d)-[rel:{r['relationship']}]->(dis)
                SET rel.source_chunk_ids = $source_chunk_ids
                """,
                chem_norm=r["chemical_normalized"],
                chem_display=r["chemical"],
                dis_norm=r["disease_normalized"],
                dis_display=r["disease"],
                source_chunk_ids=r["source_chunk_ids"],
            )
    print(f"Wrote {len(relationships)} relationships")


setup_constraints(driver)
write_relationships(driver, final_relationships)

Constraints created
Wrote 19 relationships


# Query back and verify the graph

In [11]:
with driver.session() as session:
    result = session.run("""
        MATCH (d:Drug)-[rel]->(dis:Disease)
        RETURN d.name AS drug, type(rel) AS relationship, dis.name AS disease, rel.source_chunk_ids AS evidence
        ORDER BY relationship, disease
    """)
    for record in result:
        print(f"({record['drug']}) --[{record['relationship']}]--> ({record['disease']})   evidence={len(record['evidence'])} chunk(s)")

(Metformin Hydrochloride) --[CAUSES]--> (Hypoglycemia)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (Lactic Acidosis)   evidence=2 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (abdominal pain)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (acidosis)   evidence=2 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (bradyarrhythmias)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (hypotension)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (ketonemia)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (ketonuria)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (metformin-associated lactic acidosis)   evidence=3 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (myalgias)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (renal impairment)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (respiratory distress)   evidence=1 chunk(s)
(Metformin Hydrochloride) --[CAUSES]--> (somno

# Full-scale extraction and write across all OpenFDA chunks

In [12]:
all_openfda_chunks_full = list(load_chunks(source="openfda", topic=None, output_dir=CHUNKS_DIR)) \
    if False else None  # placeholder - load_chunks requires a topic; see loader below

# load_chunks requires a topic param - load across every OpenFDA topic file on disk,
# deduped by chunk_id, same pattern used in Phase 8/9's load_all_chunks_for_source
def load_all_openfda_chunks(chunks_dir: str) -> list:
    source_dir = Path(chunks_dir) / "openfda"
    seen_ids = set()
    chunks = []
    for filepath in sorted(source_dir.glob("*.jsonl")):
        topic = filepath.stem
        for chunk in load_chunks(source="openfda", topic=topic, output_dir=chunks_dir):
            if chunk.chunk_id not in seen_ids:
                seen_ids.add(chunk.chunk_id)
                chunks.append(chunk)
    return chunks

all_openfda_chunks_full = load_all_openfda_chunks(CHUNKS_DIR)
print(f"Total unique OpenFDA chunks: {len(all_openfda_chunks_full)}")

# extract relationships from every chunk
all_relationships_full = []
for i, chunk in enumerate(all_openfda_chunks_full):
    rels = extract_relationships_from_chunk_v3(chunk)
    all_relationships_full.extend(rels)
    if (i + 1) % 1000 == 0:
        print(f"  processed {i + 1}/{len(all_openfda_chunks_full)} chunks...")

print(f"\nTotal raw relationships before aggregation: {len(all_relationships_full)}")

final_relationships_full = aggregate_relationships(all_relationships_full)
print(f"Final unique relationships after aggregation: {len(final_relationships_full)}")

Total unique OpenFDA chunks: 13167
  processed 1000/13167 chunks...
  processed 2000/13167 chunks...
  processed 3000/13167 chunks...
  processed 4000/13167 chunks...
  processed 5000/13167 chunks...
  processed 6000/13167 chunks...
  processed 7000/13167 chunks...
  processed 8000/13167 chunks...
  processed 9000/13167 chunks...
  processed 10000/13167 chunks...
  processed 11000/13167 chunks...
  processed 12000/13167 chunks...
  processed 13000/13167 chunks...

Total raw relationships before aggregation: 79727
Final unique relationships after aggregation: 49133


# Batch-write the full relationship set to Neo4j

In [13]:
def write_relationships_batched(driver, relationships: list, batch_size: int = 500):
    with driver.session() as session:
        for i in range(0, len(relationships), batch_size):
            batch = relationships[i:i + batch_size]

            # group by relationship type within this batch, since Cypher
            # can't parameterize relationship types (see Cell 10's note)
            by_type = defaultdict(list)
            for r in batch:
                by_type[r["relationship"]].append(r)

            for rel_type, rows in by_type.items():
                session.run(
                    f"""
                    UNWIND $rows AS row
                    MERGE (d:Drug {{normalized_name: row.chemical_normalized}})
                    ON CREATE SET d.name = row.chemical
                    MERGE (dis:Disease {{normalized_name: row.disease_normalized}})
                    ON CREATE SET dis.name = row.disease
                    MERGE (d)-[rel:{rel_type}]->(dis)
                    SET rel.source_chunk_ids = row.source_chunk_ids
                    """,
                    rows=rows,
                )
            print(f"  wrote {min(i + batch_size, len(relationships))}/{len(relationships)}")


write_relationships_batched(driver, final_relationships_full)

  wrote 500/49133
  wrote 1000/49133
  wrote 1500/49133
  wrote 2000/49133
  wrote 2500/49133
  wrote 3000/49133
  wrote 3500/49133
  wrote 4000/49133
  wrote 4500/49133
  wrote 5000/49133
  wrote 5500/49133
  wrote 6000/49133
  wrote 6500/49133
  wrote 7000/49133
  wrote 7500/49133
  wrote 8000/49133
  wrote 8500/49133
  wrote 9000/49133
  wrote 9500/49133
  wrote 10000/49133
  wrote 10500/49133
  wrote 11000/49133
  wrote 11500/49133
  wrote 12000/49133
  wrote 12500/49133
  wrote 13000/49133
  wrote 13500/49133
  wrote 14000/49133
  wrote 14500/49133
  wrote 15000/49133
  wrote 15500/49133
  wrote 16000/49133
  wrote 16500/49133
  wrote 17000/49133
  wrote 17500/49133
  wrote 18000/49133
  wrote 18500/49133
  wrote 19000/49133
  wrote 19500/49133
  wrote 20000/49133
  wrote 20500/49133
  wrote 21000/49133
  wrote 21500/49133
  wrote 22000/49133
  wrote 22500/49133
  wrote 23000/49133
  wrote 23500/49133
  wrote 24000/49133
  wrote 24500/49133
  wrote 25000/49133
  wrote 25500/49133


# Verify final graph counts

In [14]:
with driver.session() as session:
    drug_count = session.run("MATCH (d:Drug) RETURN count(d) AS count").single()["count"]
    disease_count = session.run("MATCH (dis:Disease) RETURN count(dis) AS count").single()["count"]
    rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS count").single()["count"]

    rel_by_type = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS relationship, count(r) AS count
        ORDER BY count DESC
    """)

    print(f"Drug nodes: {drug_count}")
    print(f"Disease nodes: {disease_count}")
    print(f"Total relationships: {rel_count}")
    print("\nBy type:")
    for record in rel_by_type:
        print(f"  {record['relationship']}: {record['count']}")

Drug nodes: 466
Disease nodes: 7069
Total relationships: 49133

By type:
  CAUSES: 42206
  TREATS: 5125
  CONTRAINDICATED_IN: 1802


# Investigate the missing drug

In [15]:
# Get the set of all 467 known drug source_ids from the chunk data
all_drug_ids = set(c.source_id for c in all_openfda_chunks_full)
print(f"Unique drug source_ids in chunk data: {len(all_drug_ids)}")

# Get the set of normalized drug names actually in Neo4j
with driver.session() as session:
    result = session.run("MATCH (d:Drug) RETURN d.normalized_name AS name")
    graph_drug_names = set(record["name"] for record in result)

# Which source_ids never made it into the graph?
missing = [drug_id for drug_id in all_drug_ids if normalize_entity(drug_id) not in graph_drug_names]
print(f"Drugs missing from graph: {missing}")

# Check: any normalization collisions? (two different drug_ids -> same normalized name)
from collections import Counter
norm_counts = Counter(normalize_entity(drug_id) for drug_id in all_drug_ids)
collisions = {name: count for name, count in norm_counts.items() if count > 1}
print(f"Normalization collisions: {collisions}")

Unique drug source_ids in chunk data: 467
Drugs missing from graph: ['R Cos']
Normalization collisions: {}


# Confirm why 'R Cos' has no relationships

In [16]:
r_cos_chunks = [c for c in all_openfda_chunks_full if c.source_id == "R Cos"]
print(f"'R Cos' has {len(r_cos_chunks)} chunks")
for c in r_cos_chunks:
    print(f"  chunk_id={c.chunk_id}  field={c.metadata.get('field')}")

'R Cos' has 2 chunks
  chunk_id=R Cos_openfda_0  field=indications_and_usage
  chunk_id=R Cos_openfda_1  field=dosage_and_administration


# Inspect the actual text

In [17]:
for c in r_cos_chunks:
    if c.metadata.get("field") == "indications_and_usage":
        print(f"chunk_id={c.chunk_id}")
        print(f"raw_text: {c.raw_text}")
        result = extract_medical_entities(c.raw_text)
        print(f"extracted: {result}")

chunk_id=R Cos_openfda_0
raw_text: Uses A dietary supplement to support the immune system and help alleviate symptoms associated with various viral and bacterial infections, such as those caused by COVID-19 and MERS. Depending on the stmptoms, it may be taken alongside antipyretics, expectorants, and analgesics.
extracted: {'chemicals': [], 'diseases': [], 'dosages': []}
